<a href="https://colab.research.google.com/github/AlanIslasZd/CSQL_by_Rep/blob/main/stage_velocity_analysis_Jan_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Source

```sql
-- Stage Timeline Analysis: First/Last Entry per Stage per Opp
-- Track when each opp entered and exited each stage in sequence

WITH qualified_opps AS (
    -- Only include opps that EVER reached numbered Stage 02-08 (excludes opps that only have Lost/Failed Finance Audit)
    SELECT DISTINCT ID
    FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_SCD2
    WHERE CAMPAIGN_ID LIKE '%70180000001JlouAAC%'
        AND CREATED_DATE > '2024-12-31'
        AND STAGE_NAME IN ('02 - Confirm Need', '03 - Establish Value', '04 - Demonstrate Value',
                           '05 - Secure Commitment', '06 - Contracting', '07 - Signed', '08 - Closed')
),
opp_snapshot AS (
    -- Get current snapshot with created/close dates and outcome
    SELECT
        o.ID,
        DATE(o.CREATED_DATE) AS CREATED_DATE,
        o.CLOSE_DATE,
        o.IS_WON,
        CASE
            WHEN o.IS_WON = TRUE THEN 'WON'
            WHEN o.STAGE_NAME IN ('08 - Closed', 'Lost', 'Failed Finance Audit') THEN 'LOST'
            ELSE 'OPEN'
        END AS OUTCOME,
        CASE
            WHEN o.IS_WON = TRUE THEN DATEDIFF('day', DATE(o.CREATED_DATE), o.CLOSE_DATE)
            WHEN o.STAGE_NAME IN ('08 - Closed', 'Lost', 'Failed Finance Audit') THEN DATEDIFF('day', DATE(o.CREATED_DATE), o.CLOSE_DATE)
            ELSE NULL
        END AS TIME_TO_CLOSE,
        CASE
            WHEN o.IS_WON = TRUE THEN DATE(o.CLOSE_DATE)
            ELSE NULL
        END AS DATE_WON,
        CASE
            WHEN o.IS_WON = FALSE AND o.STAGE_NAME IN ('08 - Closed', 'Lost', 'Failed Finance Audit') THEN DATE(o.CLOSE_DATE)
            ELSE NULL
        END AS DATE_LOST
    FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_SCD2 o
    JOIN qualified_opps q ON o.ID = q.ID
    WHERE o.VALID_TO_TIMESTAMP = '9999-12-31'
        AND o.CAMPAIGN_ID LIKE '%70180000001JlouAAC%'
        AND o.CREATED_DATE > '2024-12-31'
),
stage_entries AS (
    -- Get all stage transitions with timestamps (only for qualified opps, only Stage 02+)
    SELECT
        se.ID,
        se.STAGE_NAME,
        se.VALID_FROM_TIMESTAMP,
        se.VALID_TO_TIMESTAMP,
        -- Rank to find FIRST entry into each stage
        ROW_NUMBER() OVER (PARTITION BY se.ID, se.STAGE_NAME ORDER BY se.VALID_FROM_TIMESTAMP ASC) AS RN_FIRST,
        -- Rank to find LAST exit from each stage
        ROW_NUMBER() OVER (PARTITION BY se.ID, se.STAGE_NAME ORDER BY se.VALID_FROM_TIMESTAMP DESC) AS RN_LAST
    FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_SCD2 se
    JOIN qualified_opps q ON se.ID = q.ID
    WHERE se.CAMPAIGN_ID LIKE '%70180000001JlouAAC%'
        AND se.CREATED_DATE > '2024-12-31'
        AND se.STAGE_NAME NOT IN ('00 - Prospect & Plan', '01 - Qualify Need', 'Omitted', 'Failed Finance Audit')
),
first_last_per_stage AS (
    -- Get first entry and last exit for each stage per opp
    SELECT
        se.ID,
        se.STAGE_NAME,
        DATE(MIN(CASE WHEN RN_FIRST = 1 THEN VALID_FROM_TIMESTAMP END)) AS STAGE_FIRST_ENTRY,
        -- Replace 9999-12-31 with CLOSE_DATE for closed opps, or CURRENT_DATE for OPEN opps
        CASE
            WHEN DATE(MAX(CASE WHEN RN_LAST = 1 THEN se.VALID_TO_TIMESTAMP END)) = '9999-12-31'
            THEN COALESCE(DATE(os.CLOSE_DATE), CURRENT_DATE())  -- Use CLOSE_DATE if available, else CURRENT_DATE
            ELSE DATE(MAX(CASE WHEN RN_LAST = 1 THEN se.VALID_TO_TIMESTAMP END))
        END AS STAGE_LAST_EXIT
    FROM stage_entries se
    JOIN opp_snapshot os ON se.ID = os.ID
    GROUP BY se.ID, se.STAGE_NAME, os.CLOSE_DATE
),
stage_sequence AS (
    -- Add stage order and sequence number
    SELECT
        ID,
        STAGE_NAME,
        STAGE_FIRST_ENTRY,
        STAGE_LAST_EXIT,
        -- Order stages by first entry time to get actual journey sequence
        ROW_NUMBER() OVER (PARTITION BY ID ORDER BY STAGE_FIRST_ENTRY) AS STAGE_SEQ,
        -- Calculate days in stage (9999-12-31 already replaced with CURRENT_DATE in first_last_per_stage)
        DATEDIFF('day', STAGE_FIRST_ENTRY, STAGE_LAST_EXIT) AS DAYS_IN_STAGE
    FROM first_last_per_stage
)
SELECT
    ss.ID,
    os.CREATED_DATE,
    os.OUTCOME,
    os.TIME_TO_CLOSE,
    os.DATE_WON,
    os.DATE_LOST,
    ss.STAGE_SEQ,
    ss.STAGE_NAME,
    ss.STAGE_FIRST_ENTRY,
    ss.STAGE_LAST_EXIT,
    ss.DAYS_IN_STAGE,
    -- Get next stage info using LEAD
    LEAD(ss.STAGE_NAME) OVER (PARTITION BY ss.ID ORDER BY ss.STAGE_SEQ) AS NEXT_STAGE,
    DATE(LEAD(ss.STAGE_FIRST_ENTRY) OVER (PARTITION BY ss.ID ORDER BY ss.STAGE_SEQ)) AS NEXT_STAGE_ENTRY,
    -- Days between exiting this stage and entering next
    DATEDIFF('day', ss.STAGE_LAST_EXIT,
        LEAD(ss.STAGE_FIRST_ENTRY) OVER (PARTITION BY ss.ID ORDER BY ss.STAGE_SEQ)
    ) AS DAYS_TO_NEXT_STAGE
FROM stage_sequence ss
JOIN opp_snapshot os ON ss.ID = os.ID
ORDER BY ss.ID, ss.STAGE_SEQ;

-- ============================================================================
-- Pivoted View: One row per opp with all stages as columns
-- Only includes opps that reached Stage 02+ (qualified leads)
-- ============================================================================
WITH qualified_opps AS (
    -- Only include opps that EVER reached numbered Stage 02-08 (excludes opps that only have Lost/Failed Finance Audit)
    SELECT DISTINCT ID
    FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_SCD2
    WHERE CAMPAIGN_ID LIKE '%70180000001JlouAAC%'
        AND CREATED_DATE > '2024-12-31'
        AND STAGE_NAME IN ('02 - Confirm Need', '03 - Establish Value', '04 - Demonstrate Value',
                           '05 - Secure Commitment', '06 - Contracting', '07 - Signed', '08 - Closed')
),
opp_snapshot AS (
    -- Get current snapshot with created/close dates and outcome
    SELECT
        o.ID,
        DATE(o.CREATED_DATE) AS CREATED_DATE,
        DATE(o.CLOSE_DATE) AS CLOSE_DATE,  -- Added for stage exit calculation
        o.IS_WON,
        CASE
            WHEN o.IS_WON = TRUE THEN 'WON'
            WHEN o.STAGE_NAME IN ('08 - Closed', 'Lost', 'Failed Finance Audit') THEN 'LOST'
            ELSE 'OPEN'
        END AS OUTCOME,
        CASE
            WHEN o.IS_WON = TRUE THEN DATEDIFF('day', DATE(o.CREATED_DATE), o.CLOSE_DATE)
            WHEN o.STAGE_NAME IN ('08 - Closed', 'Lost', 'Failed Finance Audit') THEN DATEDIFF('day', DATE(o.CREATED_DATE), o.CLOSE_DATE)
            ELSE NULL
        END AS TIME_TO_CLOSE,
        CASE
            WHEN o.IS_WON = TRUE THEN DATE(o.CLOSE_DATE)
            ELSE NULL
        END AS DATE_WON,
        CASE
            WHEN o.IS_WON = FALSE AND o.STAGE_NAME IN ('08 - Closed', 'Lost', 'Failed Finance Audit') THEN DATE(o.CLOSE_DATE)
            ELSE NULL
        END AS DATE_LOST
    FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_SCD2 o
    JOIN qualified_opps q ON o.ID = q.ID
    WHERE o.VALID_TO_TIMESTAMP = '9999-12-31'
        AND o.CAMPAIGN_ID LIKE '%70180000001JlouAAC%'
        AND o.CREATED_DATE > '2024-12-31'
),
stage_entries AS (
    SELECT
        se.ID,
        se.STAGE_NAME,
        se.VALID_FROM_TIMESTAMP,
        se.VALID_TO_TIMESTAMP,
        ROW_NUMBER() OVER (PARTITION BY se.ID, se.STAGE_NAME ORDER BY se.VALID_FROM_TIMESTAMP ASC) AS RN_FIRST,
        ROW_NUMBER() OVER (PARTITION BY se.ID, se.STAGE_NAME ORDER BY se.VALID_FROM_TIMESTAMP DESC) AS RN_LAST
    FROM CLEANSED.SALESFORCE.SALESFORCE_OPPORTUNITY_SCD2 se
    JOIN qualified_opps q ON se.ID = q.ID
    WHERE se.CAMPAIGN_ID LIKE '%70180000001JlouAAC%'
        AND se.CREATED_DATE > '2024-12-31'
        AND se.STAGE_NAME NOT IN ('00 - Prospect & Plan', '01 - Qualify Need', 'Omitted', 'Failed Finance Audit')
),
first_last_per_stage AS (
    SELECT
        se.ID,
        se.STAGE_NAME,
        DATE(MIN(CASE WHEN RN_FIRST = 1 THEN VALID_FROM_TIMESTAMP END)) AS STAGE_FIRST_ENTRY,
        -- Replace 9999-12-31 with CLOSE_DATE for closed opps, or CURRENT_DATE for OPEN opps
        CASE
            WHEN DATE(MAX(CASE WHEN RN_LAST = 1 THEN se.VALID_TO_TIMESTAMP END)) = '9999-12-31'
            THEN COALESCE(os.CLOSE_DATE, CURRENT_DATE())  -- Use CLOSE_DATE if available, else CURRENT_DATE
            ELSE DATE(MAX(CASE WHEN RN_LAST = 1 THEN se.VALID_TO_TIMESTAMP END))
        END AS STAGE_LAST_EXIT
    FROM stage_entries se
    JOIN opp_snapshot os ON se.ID = os.ID
    GROUP BY se.ID, se.STAGE_NAME, os.CLOSE_DATE
)
SELECT
    flps.ID,
    os.CREATED_DATE,
    os.OUTCOME,
    os.TIME_TO_CLOSE,
    os.DATE_WON,
    os.DATE_LOST,
    -- Stage 02
    DATE(MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END)) AS STAGE_02_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)) AS STAGE_02_LAST_EXIT,
    -- Stage 03
    DATE(MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END)) AS STAGE_03_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)) AS STAGE_03_LAST_EXIT,
    -- Stage 04
    DATE(MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END)) AS STAGE_04_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)) AS STAGE_04_LAST_EXIT,
    -- Stage 05
    DATE(MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END)) AS STAGE_05_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)) AS STAGE_05_LAST_EXIT,
    -- Stage 06
    DATE(MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END)) AS STAGE_06_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)) AS STAGE_06_LAST_EXIT,
    -- Stage 07
    DATE(MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END)) AS STAGE_07_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)) AS STAGE_07_LAST_EXIT,
    -- Stage 08
    DATE(MAX(CASE WHEN STAGE_NAME = '08 - Closed' THEN STAGE_FIRST_ENTRY END)) AS STAGE_08_FIRST_ENTRY,
    DATE(MAX(CASE WHEN STAGE_NAME = '08 - Closed' THEN STAGE_LAST_EXIT END)) AS STAGE_08_LAST_EXIT,
    
    -- ==================== DAYS IN EACH STAGE ====================
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_02,
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_03,
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_04,
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_05,
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_06,
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_07,
    DATEDIFF('day',
        MAX(CASE WHEN STAGE_NAME = '08 - Closed' THEN STAGE_FIRST_ENTRY END),
        MAX(CASE WHEN STAGE_NAME = '08 - Closed' THEN STAGE_LAST_EXIT END)
    ) AS DAYS_IN_STAGE_08,
    
    -- ==================== STAGE JOURNEY METRICS ====================
    -- Number of stages visited (non-null) - EXCLUDING Stage 08 (terminal state)
    (CASE WHEN MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN 1 END) = 1 THEN 1 ELSE 0 END +
     CASE WHEN MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN 1 END) = 1 THEN 1 ELSE 0 END +
     CASE WHEN MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN 1 END) = 1 THEN 1 ELSE 0 END +
     CASE WHEN MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN 1 END) = 1 THEN 1 ELSE 0 END +
     CASE WHEN MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN 1 END) = 1 THEN 1 ELSE 0 END +
     CASE WHEN MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN 1 END) = 1 THEN 1 ELSE 0 END
    ) AS NUM_STAGES_VISITED,
    
    -- First stage (earliest non-null)
    CASE
        WHEN MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '02'
        WHEN MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '03'
        WHEN MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '04'
        WHEN MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '05'
        WHEN MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '06'
        WHEN MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '07'
        WHEN MAX(CASE WHEN STAGE_NAME = '08 - Closed' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '08'
    END AS FIRST_STAGE,
    
    -- Last stage (latest non-null)
    CASE
        WHEN MAX(CASE WHEN STAGE_NAME = '08 - Closed' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '08'
        WHEN MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '07'
        WHEN MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '06'
        WHEN MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '05'
        WHEN MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '04'
        WHEN MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '03'
        WHEN MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END) IS NOT NULL THEN '02'
    END AS LAST_STAGE,
    
    -- Days in first stage (first non-null duration) - EXCLUDING Stage 08
    COALESCE(
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END))
    ) AS DAYS_IN_FIRST_STAGE,
    
    -- Days in last stage (last non-null duration) - EXCLUDING Stage 08
    COALESCE(
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)),
        DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END))
    ) AS DAYS_IN_LAST_STAGE,
    
    -- Total days in pipeline (sum of all stage durations) - EXCLUDING Stage 08 (terminal state)
    COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)), 0) +
    COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)), 0) +
    COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)), 0) +
    COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)), 0) +
    COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)), 0) +
    COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)), 0)
    AS TOTAL_DAYS_IN_PIPELINE,
    
    -- Bottleneck stage (stage with max duration) - EXCLUDING Stage 08 (terminal state)
    CASE GREATEST(
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)), -1),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)), -1),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)), -1),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)), -1),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)), -1),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)), -1)
    )
        WHEN COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)), -1) THEN '02'
        WHEN COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)), -1) THEN '03'
        WHEN COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)), -1) THEN '04'
        WHEN COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)), -1) THEN '05'
        WHEN COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)), -1) THEN '06'
        WHEN COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)), -1) THEN '07'
    END AS BOTTLENECK_STAGE,
    
    -- Max days in any stage (bottleneck duration) - EXCLUDING Stage 08
    GREATEST(
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '02 - Confirm Need' THEN STAGE_LAST_EXIT END)), 0),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '03 - Establish Value' THEN STAGE_LAST_EXIT END)), 0),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '04 - Demonstrate Value' THEN STAGE_LAST_EXIT END)), 0),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '05 - Secure Commitment' THEN STAGE_LAST_EXIT END)), 0),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '06 - Contracting' THEN STAGE_LAST_EXIT END)), 0),
        COALESCE(DATEDIFF('day', MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_FIRST_ENTRY END), MAX(CASE WHEN STAGE_NAME = '07 - Signed' THEN STAGE_LAST_EXIT END)), 0)
    ) AS BOTTLENECK_DAYS

FROM first_last_per_stage flps
JOIN opp_snapshot os ON flps.ID = os.ID
GROUP BY flps.ID, os.CREATED_DATE, os.OUTCOME, os.TIME_TO_CLOSE, os.DATE_WON, os.DATE_LOST
ORDER BY flps.ID;


# 🚀 Stage Velocity Analysis - CSQL Q1 2025 Cohort

This notebook provides comprehensive analysis of opportunity stage velocity, bottlenecks, and idle patterns.

## Key Metrics Analyzed:
- **Velocity**: Time to close, days per stage, total pipeline duration
- **Bottlenecks**: Which stages slow down deals the most
- **Idle Analysis**: Opportunities stuck in stages too long
- **Win Rate**: Conversion patterns by cohort and attributes

## Sections:
1. Data Loading & Preparation
2. Univariate Analysis
3. Bottleneck Analysis
4. Idle Indicators & Backlog Health
5. Velocity Trends Over Time
6. Win Rate Analysis
7. Actionable Insights

In [230]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Plotly default template with dark background
PLOT_TEMPLATE = 'plotly_dark'
COLOR_MAP = {'WON': '#A1D78F', 'LOST': '#FEEB7E', 'OPEN': '#2D4C33'}
STAGE_COLORS = px.colors.qualitative.Set2

## 1. Data Loading & Preparation

In [231]:
df = pd.read_csv('b8210223356a.csv')


In [232]:
df.head().T

,0,1,2,3,4
ID,006PC00000GDVRNYA5,006PC00000GEwcPYAT,006PC00000GPMPCYA5,006PC00000GPf8RYAT,006PC00000GS42fYAD
CREATED_DATE,2024-12-31,2025-01-01,2025-01-07,2025-01-07,2025-01-08
OUTCOME,LOST,WON,WON,WON,WON
TIME_TO_CLOSE,107.0,356.0,83.0,-1.0,232.0
DATE_WON,NaN,2025-12-23,2025-03-31,2025-01-06,2025-08-28
DATE_LOST,2025-04-17,NaN,NaN,NaN,NaN
STAGE_02_FIRST_ENTRY,2024-12-31,2025-08-14,NaN,NaN,2025-02-06
STAGE_02_LAST_EXIT,2025-04-17,2025-11-11,NaN,NaN,2025-04-16
STAGE_03_FIRST_ENTRY,2025-01-31,2025-11-11,NaN,NaN,2025-04-16
STAGE_03_LAST_EXIT,2025-02-03,2025-12-20,NaN,NaN,2025-07-17


In [233]:

# Convert date columns
date_cols = ['CREATED_DATE', 'DATE_WON', 'DATE_LOST'] + [col for col in df.columns if 'ENTRY' in col or 'EXIT' in col]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Extract time dimensions from CREATED_DATE
df['CREATED_MONTH'] = df['CREATED_DATE'].dt.to_period('M').astype(str)
df['CREATED_QUARTER'] = df['CREATED_DATE'].dt.to_period('Q').astype(str)
df['CREATED_WEEK'] = df['CREATED_DATE'].dt.to_period('W').astype(str)

print(f"✅ Loaded {len(df):,} opportunities")
print(f"\n📊 Outcome Distribution:")
print(df['OUTCOME'].value_counts())

✅ Loaded 2,473 opportunities

📊 Outcome Distribution:
OUTCOME
LOST    950
OPEN    934
WON     589
Name: count, dtype: int64


In [234]:
# ==================== VELOCITY INDICATORS ====================
# Determine close date for each opp (DATE_WON or DATE_LOST)
df['CLOSE_DATE'] = df['DATE_WON'].combine_first(df['DATE_LOST'])

# SAME_QTR_CLOSE: Created and closed in the same quarter
df['CREATED_QTR'] = df['CREATED_DATE'].dt.to_period('Q')
df['CLOSED_QTR'] = df['CLOSE_DATE'].dt.to_period('Q')
df['SAME_QTR_CLOSE'] = (df['CREATED_QTR'] == df['CLOSED_QTR']) & df['CLOSE_DATE'].notna()

# FAST_CLOSE: Closed within 60 days of creation
df['FAST_CLOSE'] = (df['TIME_TO_CLOSE'] <= 60) & df['TIME_TO_CLOSE'].notna()

print("\n✅ Velocity indicators created:")
print(f"   - SAME_QTR_CLOSE: Created & closed in same quarter")
print(f"   - FAST_CLOSE: Closed within 60 days")

# Summary for closed opps
closed_mask = df['OUTCOME'].isin(['WON', 'LOST'])
print(f"\n📊 Velocity Summary (Closed Opps Only):")
print(f"   Same Quarter Close: {df.loc[closed_mask, 'SAME_QTR_CLOSE'].sum():,} / {closed_mask.sum():,} ({df.loc[closed_mask, 'SAME_QTR_CLOSE'].mean()*100:.1f}%)")
print(f"   Fast Close (≤60d):  {df.loc[closed_mask, 'FAST_CLOSE'].sum():,} / {closed_mask.sum():,} ({df.loc[closed_mask, 'FAST_CLOSE'].mean()*100:.1f}%)")

# ==================== IDLE INDICATORS ====================
# Define idle thresholds
def categorize_idle_days(days):
    """Categorize days into idle bins"""
    if pd.isna(days) or days < 0:
        return None
    elif days <= 15:
        return '0-15 days'
    elif days <= 30:
        return '15-30 days'
    elif days <= 60:
        return '1-2 months'
    elif days <= 90:
        return '2-3 months'
    else:
        return '3+ months'

# Calculate days since last stage activity for OPEN opps
df['DAYS_SINCE_LAST_ACTIVITY'] = None
for idx, row in df[df['OUTCOME'] == 'OPEN'].iterrows():
    # Find the most recent stage exit date
    exit_cols = [col for col in df.columns if 'LAST_EXIT' in col]
    exit_dates = [row[col] for col in exit_cols if pd.notna(row[col])]
    if exit_dates:
        last_exit = max(exit_dates)
        days_idle = (pd.Timestamp.now() - last_exit).days
        df.loc[idx, 'DAYS_SINCE_LAST_ACTIVITY'] = days_idle

# Idle stage = current stage (LAST_STAGE) for OPEN opps
df['IDLE_STAGE'] = df.apply(lambda x: x['LAST_STAGE'] if x['OUTCOME'] == 'OPEN' else None, axis=1)

# Idle days = DAYS_IN_LAST_STAGE for OPEN opps (how long they've been in current stage)
df['IDLE_DAYS'] = df.apply(lambda x: x['DAYS_IN_LAST_STAGE'] if x['OUTCOME'] == 'OPEN' else None, axis=1)

# Idle bins
df['IDLE_BIN'] = df['IDLE_DAYS'].apply(categorize_idle_days)

# Bottleneck bins (for all opps)
df['BOTTLENECK_BIN'] = df['BOTTLENECK_DAYS'].apply(categorize_idle_days)
df['BOTTLENECK_DAYS'] = df['BOTTLENECK_DAYS'].clip(upper=250)

# Average duration per stage
df['AVG_DAYS_PER_STAGE'] = df['TOTAL_DAYS_IN_PIPELINE'] / df['NUM_STAGES_VISITED']

print("✅ Idle indicators created:")
print(f"   - IDLE_STAGE: Current stage for OPEN opps")
print(f"   - IDLE_DAYS: Days in current stage for OPEN opps")
print(f"   - IDLE_BIN: Categorized idle duration")
print(f"   - BOTTLENECK_BIN: Categorized bottleneck duration")
print(f"   - AVG_DAYS_PER_STAGE: Average duration per stage visited")

print(f"\n📊 OPEN Opps Idle Distribution:")
print(df[df['OUTCOME'] == 'OPEN']['IDLE_BIN'].value_counts())


✅ Velocity indicators created:
   - SAME_QTR_CLOSE: Created & closed in same quarter
   - FAST_CLOSE: Closed within 60 days

📊 Velocity Summary (Closed Opps Only):
   Same Quarter Close: 606 / 1,539 (39.4%)
   Fast Close (≤60d):  721 / 1,539 (46.8%)
✅ Idle indicators created:
   - IDLE_STAGE: Current stage for OPEN opps
   - IDLE_DAYS: Days in current stage for OPEN opps
   - IDLE_BIN: Categorized idle duration
   - BOTTLENECK_BIN: Categorized bottleneck duration
   - AVG_DAYS_PER_STAGE: Average duration per stage visited

📊 OPEN Opps Idle Distribution:
IDLE_BIN
3+ months     635
2-3 months    123
1-2 months    100
15-30 days     42
0-15 days      24
Name: count, dtype: int64


## 2. Univariate Analysis - Overview Statistics

In [235]:
# Summary statistics by OUTCOME
print("="*80)
print("📊 SUMMARY STATISTICS BY OUTCOME")
print("="*80)

for outcome in ['WON', 'LOST', 'OPEN']:
    subset = df[df['OUTCOME'] == outcome]
    print(f"\n🔹 {outcome} ({len(subset):,} opps):")
    print(f"   Median TIME_TO_CLOSE: {subset['TIME_TO_CLOSE'].median():.0f} days" if outcome != 'OPEN' else "   TIME_TO_CLOSE: N/A (still open)")
    print(f"   Median TOTAL_DAYS_IN_PIPELINE: {subset['TOTAL_DAYS_IN_PIPELINE'].median():.0f} days")
    print(f"   Median NUM_STAGES_VISITED: {subset['NUM_STAGES_VISITED'].median():.0f} stages")
    print(f"   Median AVG_DAYS_PER_STAGE: {subset['AVG_DAYS_PER_STAGE'].median():.0f} days")
    print(f"   Top BOTTLENECK_STAGE: {subset['BOTTLENECK_STAGE'].mode().values[0] if len(subset['BOTTLENECK_STAGE'].mode()) > 0 else 'N/A'}")
    print(f"   Median BOTTLENECK_DAYS: {subset['BOTTLENECK_DAYS'].median():.0f} days")

📊 SUMMARY STATISTICS BY OUTCOME

🔹 WON (589 opps):
   Median TIME_TO_CLOSE: 48 days
   Median TOTAL_DAYS_IN_PIPELINE: 26 days
   Median NUM_STAGES_VISITED: 2 stages
   Median AVG_DAYS_PER_STAGE: 12 days
   Top BOTTLENECK_STAGE: 2
   Median BOTTLENECK_DAYS: 19 days

🔹 LOST (950 opps):
   Median TIME_TO_CLOSE: 76 days
   Median TOTAL_DAYS_IN_PIPELINE: 47 days
   Median NUM_STAGES_VISITED: 1 stages
   Median AVG_DAYS_PER_STAGE: 36 days
   Top BOTTLENECK_STAGE: 2
   Median BOTTLENECK_DAYS: 41 days

🔹 OPEN (934 opps):
   TIME_TO_CLOSE: N/A (still open)
   Median TOTAL_DAYS_IN_PIPELINE: 149 days
   Median NUM_STAGES_VISITED: 1 stages
   Median AVG_DAYS_PER_STAGE: 116 days
   Top BOTTLENECK_STAGE: 2
   Median BOTTLENECK_DAYS: 134 days


## 3. Bottleneck Analysis

In [236]:
# Bottleneck Stage Value Counts - Bar Plot
fig = px.histogram(
    df,
    x='BOTTLENECK_STAGE',
    color='OUTCOME',
    color_discrete_map=COLOR_MAP,
    title='🚧 Bottleneck Stage Distribution by Outcome',
    labels={'BOTTLENECK_STAGE': 'Bottleneck Stage', 'count': 'Number of Opps'},
    barmode='group',
    category_orders={'BOTTLENECK_STAGE': ['02', '03', '04', '05', '06', '07', '08']}
)
fig.update_layout(
    template=PLOT_TEMPLATE,
    xaxis_title='Bottleneck Stage',
    yaxis_title='Number of Opportunities',
    legend_title='Outcome'
)
fig.show()

In [237]:
# Bottleneck Days Distribution by Outcome
fig = px.box(
    df,
    x='OUTCOME',
    y='BOTTLENECK_DAYS',
    color='OUTCOME',
    color_discrete_map=COLOR_MAP,
    title='📦 Bottleneck Days Distribution by Outcome',
    labels={'BOTTLENECK_DAYS': 'Days in Bottleneck Stage', 'OUTCOME': 'Outcome'}
)
fig.update_layout(
    template=PLOT_TEMPLATE,
    showlegend=False
)
fig.show()

# Also show histogram
fig2 = px.histogram(
    df,
    x='BOTTLENECK_DAYS',
    color='OUTCOME',
    color_discrete_map=COLOR_MAP,
    title='📊 Bottleneck Days Histogram by Outcome',
    nbins=50,
    marginal='box',
    opacity=0.7
)
fig2.update_layout(template=PLOT_TEMPLATE, barmode='overlay')
fig2.show()

In [238]:
# Distribution of Days in First Stage vs Days in Last Stage (Overlapped)
# With OUTCOME dropdown filter using Plotly buttons (Colab-compatible)

fig = go.Figure()

# Add traces for each outcome (all hidden initially except WON)
for outcome in ['WON', 'LOST', 'OPEN']:
    plot_df = df[df['OUTCOME'] == outcome]
    visible = (outcome == 'WON')  # Only WON visible by default

    # Days in First Stage
    fig.add_trace(go.Histogram(
        x=plot_df['DAYS_IN_FIRST_STAGE'],
        name=f'Days in First Stage',
        opacity=0.6,
        marker_color='#636EFA',
        nbinsx=50,
        visible=visible,
        legendgroup=outcome
    ))

    # Days in Last Stage
    fig.add_trace(go.Histogram(
        x=plot_df['DAYS_IN_LAST_STAGE'],
        name=f'Days in Last Stage',
        opacity=0.6,
        marker_color='#EF553B',
        nbinsx=50,
        visible=visible,
        legendgroup=outcome
    ))

# Create dropdown buttons
buttons = []
outcomes = ['WON', 'LOST', 'OPEN']
for i, outcome in enumerate(outcomes):
    # Each outcome has 2 traces (first stage, last stage)
    visibility = [False] * 6  # 3 outcomes x 2 traces each
    visibility[i*2] = True     # First stage trace
    visibility[i*2 + 1] = True # Last stage trace

    plot_df = df[df['OUTCOME'] == outcome]
    buttons.append(
        dict(
            label=f"{outcome} (n={len(plot_df):,})",
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'📊 Distribution: Days in First Stage vs Last Stage ({outcome} Opps, n={len(plot_df):,})'}
            ]
        )
    )

fig.update_layout(
    title=f'📊 Distribution: Days in First Stage vs Last Stage (WON Opps, n={len(df[df["OUTCOME"]=="WON"]):,})',
    xaxis_title='Days',
    yaxis_title='Count',
    barmode='overlay',
    template=PLOT_TEMPLATE,
    legend=dict(x=0.7, y=0.95),
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.0,
            xanchor='left',
            y=1.15,
            yanchor='top',
            bgcolor='#2a2a2a',
            font=dict(color='white')
        )
    ]
)

fig.show()

# Summary stats for all outcomes
print("\n📈 Comparison Stats by Outcome:")
for outcome in ['WON', 'LOST', 'OPEN']:
    subset = df[df['OUTCOME'] == outcome]
    print(f"\n   {outcome} (n={len(subset):,}):")
    print(f"      Days in First Stage - Median: {subset['DAYS_IN_FIRST_STAGE'].median():.0f}, Mean: {subset['DAYS_IN_FIRST_STAGE'].mean():.0f}")
    print(f"      Days in Last Stage  - Median: {subset['DAYS_IN_LAST_STAGE'].median():.0f}, Mean: {subset['DAYS_IN_LAST_STAGE'].mean():.0f}")


📈 Comparison Stats by Outcome:

   WON (n=589):
      Days in First Stage - Median: 13, Mean: 23
      Days in Last Stage  - Median: 3, Mean: 8

   LOST (n=950):
      Days in First Stage - Median: 33, Mean: 47
      Days in Last Stage  - Median: 36, Mean: 49

   OPEN (n=934):
      Days in First Stage - Median: 103, Mean: 123
      Days in Last Stage  - Median: 128, Mean: 151


## 4. Idle Analysis & Backlog Health

In [239]:
# Filter to OPEN opps only for idle analysis
open_df = df[df['OUTCOME'] == 'OPEN'].copy()

print(f"📊 BACKLOG HEALTH CHECK: {len(open_df):,} OPEN Opportunities")
print("="*60)

# Idle Bin Distribution
fig = px.pie(
    open_df,
    names='IDLE_BIN',
    title='🕐 OPEN Opps: Time in Current Stage (Idle Distribution)',
    color_discrete_sequence=px.colors.diverging.RdYlGn[::-1],
    hole=0.4
)
fig.update_layout(template=PLOT_TEMPLATE)
fig.show()

📊 BACKLOG HEALTH CHECK: 934 OPEN Opportunities


In [240]:
# Where are OPEN opps stuck? (Idle Stage breakdown)
idle_stage_counts = open_df.groupby(['IDLE_STAGE', 'IDLE_BIN']).size().reset_index(name='COUNT')

fig = px.bar(
    idle_stage_counts,
    x='IDLE_STAGE',
    y='COUNT',
    color='IDLE_BIN',
    title='🚧 Where Are OPEN Opps Stuck? (Stage x Idle Duration)',
    labels={'IDLE_STAGE': 'Current Stage', 'COUNT': 'Number of Opps', 'IDLE_BIN': 'Time in Stage'},
    category_orders={
        'IDLE_STAGE': ['02', '03', '04', '05', '06', '07', '08'],
        'IDLE_BIN': ['0-15 days', '15-30 days', '1-2 months', '2-3 months', '3+ months']
    },
    color_discrete_sequence=px.colors.sequential.OrRd
)
fig.update_layout(template=PLOT_TEMPLATE, barmode='stack')
fig.show()

# Summary table
print("\n📋 OPEN Opps by Current Stage:")
print(open_df.groupby('IDLE_STAGE').agg({
    'ID': 'count',
    'IDLE_DAYS': ['median', 'mean', 'max']
}).round(0))


📋 OPEN Opps by Current Stage:
              ID IDLE_DAYS              
           count    median   mean    max
IDLE_STAGE                              
2.0          473     137.0  158.0  896.0
3.0          152     150.0  164.0  484.0
4.0          191     124.0  153.0  657.0
5.0           95      73.0  100.0  470.0
6.0           22      45.0  100.0  942.0
7.0            1      54.0   54.0   54.0


In [241]:
# High-Risk Idle Opps (3+ months in current stage)
high_risk_idle = open_df[open_df['IDLE_BIN'] == '3+ months'].copy()

print(f"\n🚨 HIGH RISK: {len(high_risk_idle):,} OPEN opps idle for 3+ months")
print("="*60)

if len(high_risk_idle) > 0:
    fig = px.bar(
        high_risk_idle.groupby('IDLE_STAGE').size().reset_index(name='COUNT'),
        x='IDLE_STAGE',
        y='COUNT',
        title=f'🚨 HIGH RISK: {len(high_risk_idle)} Opps Idle 3+ Months by Stage',
        color='COUNT',
        color_continuous_scale='Reds'
    )
    fig.update_layout(template=PLOT_TEMPLATE)
    fig.show()

    print("\n📋 Sample of High-Risk Idle Opps:")
    display(high_risk_idle[['ID', 'CREATED_DATE', 'IDLE_STAGE', 'IDLE_DAYS', 'TOTAL_DAYS_IN_PIPELINE']].head(10))


🚨 HIGH RISK: 635 OPEN opps idle for 3+ months



📋 Sample of High-Risk Idle Opps:


,ID,CREATED_DATE,IDLE_STAGE,IDLE_DAYS,TOTAL_DAYS_IN_PIPELINE
13,006PC00000Gr5JhYAJ,2025-01-20,3.0,209.0,322
17,006PC00000GwEFzYAN,2025-01-22,4.0,459.0,722
20,006PC00000GxMw9YAF,2025-01-22,5.0,470.0,511
22,006PC00000H0SqXYAV,2025-01-24,4.0,280.0,479
23,006PC00000H0yLlYAJ,2025-01-24,4.0,333.0,392
32,006PC00000H8vFRYAZ,2025-01-28,4.0,265.0,315
52,006PC00000HF5jzYAD,2025-01-30,3.0,318.0,318
53,006PC00000HFDPoYAP,2025-01-30,2.0,310.0,310
60,006PC00000HGMNlYAP,2025-01-31,3.0,424.0,424
64,006PC00000HGsHBYA1,2025-01-31,5.0,167.0,244


## 5. Velocity Trends Over Time

In [242]:
# Median Time to Close by Created Month (WON opps only)
won_df = df[df['OUTCOME'] == 'WON'].copy()

monthly_velocity = won_df.groupby('CREATED_MONTH').agg({
    'TIME_TO_CLOSE': 'median',
    'ID': 'count',
    'TOTAL_DAYS_IN_PIPELINE': 'median',
    'NUM_STAGES_VISITED': 'median'
}).reset_index()
monthly_velocity.columns = ['CREATED_MONTH', 'MEDIAN_TIME_TO_CLOSE', 'WON_COUNT', 'MEDIAN_PIPELINE_DAYS', 'MEDIAN_STAGES']

# Create dual-axis chart
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=monthly_velocity['CREATED_MONTH'], y=monthly_velocity['WON_COUNT'],
           name='Won Opps', marker_color='#A1D78F', opacity=0.7),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=monthly_velocity['CREATED_MONTH'], y=monthly_velocity['MEDIAN_TIME_TO_CLOSE'],
               name='Median Time to Close', mode='lines+markers', line=dict(color='#FEEB7E', width=3)),
    secondary_y=True
)

fig.update_layout(
    title='📈 Velocity Trend: Won Opps & Median Time to Close by Created Month',
    template=PLOT_TEMPLATE,
    legend=dict(x=0.01, y=0.99)
)
fig.update_yaxes(title_text="Won Opps Count", secondary_y=False)
fig.update_yaxes(title_text="Median Days to Close", secondary_y=True)
fig.show()

print("\n📋 Monthly Velocity Summary:")
display(monthly_velocity)


📋 Monthly Velocity Summary:


,CREATED_MONTH,MEDIAN_TIME_TO_CLOSE,WON_COUNT,MEDIAN_PIPELINE_DAYS,MEDIAN_STAGES
0,2025-01,120.0,25,28.0,3.0
1,2025-02,138.0,43,58.0,3.0
2,2025-03,102.0,38,47.0,2.5
3,2025-04,144.0,34,71.0,3.0
4,2025-05,127.0,35,39.0,3.0
5,2025-06,77.5,56,32.0,2.0
6,2025-07,55.0,73,38.0,2.0
7,2025-08,39.0,65,31.0,2.0
8,2025-09,42.0,72,24.0,2.0
9,2025-10,28.0,65,19.0,2.0


In [243]:
# Pipeline Creation Trend with Backlog (OPEN opps)
monthly_creation = df.groupby(['CREATED_MONTH', 'OUTCOME']).size().reset_index(name='COUNT')
monthly_creation_pivot = monthly_creation.pivot(index='CREATED_MONTH', columns='OUTCOME', values='COUNT').fillna(0).reset_index()

fig = go.Figure()
for outcome in ['WON', 'LOST', 'OPEN']:
    if outcome in monthly_creation_pivot.columns:
        fig.add_trace(go.Bar(
            x=monthly_creation_pivot['CREATED_MONTH'],
            y=monthly_creation_pivot[outcome],
            name=outcome,
            marker_color=COLOR_MAP[outcome]
        ))

fig.update_layout(
    title='📊 Opp Creation by Month & Outcome (Backlog = OPEN)',
    xaxis_title='Created Month',
    yaxis_title='Number of Opps',
    barmode='stack',
    template=PLOT_TEMPLATE
)
fig.show()

## 6. Win Rate Analysis

In [244]:
# Win Rate by Created Month (closed opps only)
closed_df = df[df['OUTCOME'].isin(['WON', 'LOST'])].copy()

win_rate_monthly = closed_df.groupby('CREATED_MONTH').apply(
    lambda x: pd.Series({
        'WIN_RATE': (x['OUTCOME'] == 'WON').sum() / len(x) * 100,
        'TOTAL_CLOSED': len(x),
        'WON': (x['OUTCOME'] == 'WON').sum(),
        'LOST': (x['OUTCOME'] == 'LOST').sum()
    })
).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=win_rate_monthly['CREATED_MONTH'], y=win_rate_monthly['TOTAL_CLOSED'],
           name='Total Closed', marker_color='#636EFA', opacity=0.5),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=win_rate_monthly['CREATED_MONTH'], y=win_rate_monthly['WIN_RATE'],
               name='Win Rate %', mode='lines+markers', line=dict(color='#00CC96', width=3)),
    secondary_y=True
)

fig.update_layout(
    title='📈 Win Rate Trend by Created Month',
    template=PLOT_TEMPLATE
)
fig.update_yaxes(title_text="Closed Opps", secondary_y=False)
fig.update_yaxes(title_text="Win Rate %", secondary_y=True)
fig.show()

print(f"\n📊 Overall Win Rate (Closed Opps): {(closed_df['OUTCOME'] == 'WON').sum() / len(closed_df) * 100:.1f}%")
print(f"   WON: {(closed_df['OUTCOME'] == 'WON').sum():,}")
print(f"   LOST: {(closed_df['OUTCOME'] == 'LOST').sum():,}")


📊 Overall Win Rate (Closed Opps): 38.3%
   WON: 589
   LOST: 950


In [245]:
# Win Rate by Number of Stages Visited
stages_win_rate = closed_df.groupby('NUM_STAGES_VISITED').apply(
    lambda x: pd.Series({
        'WIN_RATE': (x['OUTCOME'] == 'WON').sum() / len(x) * 100,
        'COUNT': len(x)
    })
).reset_index()

fig = px.bar(
    stages_win_rate,
    x='NUM_STAGES_VISITED',
    y='WIN_RATE',
    title='📊 Win Rate by Number of Stages Visited',
    labels={'NUM_STAGES_VISITED': 'Stages Visited', 'WIN_RATE': 'Win Rate %'},
    text='COUNT',
    color='WIN_RATE',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(template=PLOT_TEMPLATE)
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.show()

In [224]:
# Win Rate by Bottleneck Stage
bottleneck_win_rate = closed_df.groupby('BOTTLENECK_STAGE').apply(
    lambda x: pd.Series({
        'WIN_RATE': (x['OUTCOME'] == 'WON').sum() / len(x) * 100,
        'COUNT': len(x),
        'MEDIAN_BOTTLENECK_DAYS': x['BOTTLENECK_DAYS'].median()
    })
).reset_index()

fig = px.bar(
    bottleneck_win_rate,
    x='BOTTLENECK_STAGE',
    y='WIN_RATE',
    title='📊 Win Rate by Bottleneck Stage',
    labels={'BOTTLENECK_STAGE': 'Bottleneck Stage', 'WIN_RATE': 'Win Rate %'},
    text='COUNT',
    color='WIN_RATE',
    color_continuous_scale='RdYlGn',
    category_orders={'BOTTLENECK_STAGE': ['02', '03', '04', '05', '06', '07', '08']}
)
fig.update_layout(template=PLOT_TEMPLATE)
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.show()

print("\n📋 Win Rate by Bottleneck Stage:")
display(bottleneck_win_rate)


📋 Win Rate by Bottleneck Stage:


,BOTTLENECK_STAGE,WIN_RATE,COUNT,MEDIAN_BOTTLENECK_DAYS
0,2,21.449704,676.0,36.0
1,3,30.593607,219.0,37.0
2,4,40.483384,331.0,35.0
3,5,68.263473,167.0,23.0
4,6,84.684685,111.0,14.0
5,7,100.000000,35.0,9.0


## 7. Actionable Insights & Next Steps

In [246]:
# ==================== ACTIONABLE INSIGHTS SUMMARY ====================
print("="*80)
print("🎯 ACTIONABLE INSIGHTS SUMMARY")
print("="*80)

# 1. Velocity Insights
print("\n📈 VELOCITY INSIGHTS:")
print(f"   • Median Time to Close (WON): {won_df['TIME_TO_CLOSE'].median():.0f} days")
print(f"   • Median Stages Visited (WON): {won_df['NUM_STAGES_VISITED'].median():.0f} stages")
print(f"   • Fastest Win: {won_df['TIME_TO_CLOSE'].min():.0f} days")
print(f"   • Slowest Win: {won_df['TIME_TO_CLOSE'].max():.0f} days")

# 2. Bottleneck Insights
print("\n🚧 BOTTLENECK INSIGHTS:")
top_bottleneck = df['BOTTLENECK_STAGE'].mode().values[0]
print(f"   • Most Common Bottleneck: Stage {top_bottleneck}")
print(f"   • Median Bottleneck Duration: {df['BOTTLENECK_DAYS'].median():.0f} days")
print(f"   • 90th Percentile Bottleneck: {df['BOTTLENECK_DAYS'].quantile(0.9):.0f} days")

# 3. Backlog Health
print("\n⚠️ BACKLOG HEALTH:")
print(f"   • Total OPEN Opps: {len(open_df):,}")
high_risk_count = len(open_df[open_df['IDLE_BIN'] == '3+ months'])
print(f"   • High Risk (3+ months idle): {high_risk_count:,} ({high_risk_count/len(open_df)*100:.1f}%)")
most_stuck_stage = open_df['IDLE_STAGE'].mode().values[0] if len(open_df) > 0 else 'N/A'
print(f"   • Most Stuck Stage: {most_stuck_stage}")

# 4. Win Rate Insights
print("\n🏆 WIN RATE INSIGHTS:")
overall_win_rate = (closed_df['OUTCOME'] == 'WON').sum() / len(closed_df) * 100
print(f"   • Overall Win Rate: {overall_win_rate:.1f}%")
best_stage_win = bottleneck_win_rate.loc[bottleneck_win_rate['WIN_RATE'].idxmax()]
print(f"   • Best Win Rate by Bottleneck: Stage {best_stage_win['BOTTLENECK_STAGE']} ({best_stage_win['WIN_RATE']:.1f}%)")

# 5. Recommendations
print("\n💡 RECOMMENDATIONS:")
print(f"   1. Focus on reducing time in Stage {top_bottleneck} (most common bottleneck)")
print(f"   2. Review {high_risk_count} OPEN opps idle >3 months for potential disqualification")
print(f"   3. Opps stuck in Stage {most_stuck_stage} need attention - most common stuck point")
print(f"   4. Target pipeline velocity < {df[df['OUTCOME']=='WON']['TIME_TO_CLOSE'].quantile(0.5):.0f} days (median for WON)")
print("="*80)

🎯 ACTIONABLE INSIGHTS SUMMARY

📈 VELOCITY INSIGHTS:
   • Median Time to Close (WON): 48 days
   • Median Stages Visited (WON): 2 stages
   • Fastest Win: -1 days
   • Slowest Win: 356 days

🚧 BOTTLENECK INSIGHTS:
   • Most Common Bottleneck: Stage 2
   • Median Bottleneck Duration: 58 days
   • 90th Percentile Bottleneck: 206 days

⚠️ BACKLOG HEALTH:
   • Total OPEN Opps: 934
   • High Risk (3+ months idle): 635 (68.0%)
   • Most Stuck Stage: 2.0

🏆 WIN RATE INSIGHTS:
   • Overall Win Rate: 38.3%
   • Best Win Rate by Bottleneck: Stage 7.0 (100.0%)

💡 RECOMMENDATIONS:
   1. Focus on reducing time in Stage 2 (most common bottleneck)
   2. Review 635 OPEN opps idle >3 months for potential disqualification
   3. Opps stuck in Stage 2.0 need attention - most common stuck point
   4. Target pipeline velocity < 48 days (median for WON)


## 📋 Future Analysis Ideas

### Additional Attributes to Add (from SQL):
- **Product**: Analyze velocity by product line
- **Region**: Compare regional performance
- **Team / Opp Owner Manager**: Identify top performers
- **Expected Revenue**: Prioritize high-value stuck opps
- **Deal Size Tier**: Does deal size impact velocity?

### Additional Questions to Explore:
1. **Stage Skip Analysis**: Which stage combinations are most commonly skipped?
2. **Regression Analysis**: What factors predict faster time-to-close?
3. **Cohort Survival Analysis**: Time-based win probability curves
4. **Seasonality**: Are there monthly/quarterly patterns in velocity?
5. **Rep Performance**: Who has best velocity with maintained win rate?
6. **Stage Transition Matrix**: Most common paths through the funnel
7. **Revenue-Weighted Velocity**: Is high-value pipeline moving differently?

### Alerts to Implement:
- 🚨 Opp idle > 30 days in any stage
- 🚨 Bottleneck duration > 90th percentile
- 🚨 Monthly win rate drop > 10%
- 🚨 Pipeline aging (avg days in pipeline increasing)

In [226]:
# Export high-risk idle opps for follow-up
if len(high_risk_idle) > 0:
    export_cols = ['ID', 'CREATED_DATE', 'OUTCOME', 'IDLE_STAGE', 'IDLE_DAYS',
                   'BOTTLENECK_STAGE', 'BOTTLENECK_DAYS', 'TOTAL_DAYS_IN_PIPELINE', 'NUM_STAGES_VISITED']
    high_risk_idle[export_cols].to_csv('high_risk_idle_opps.csv', index=False)
    print(f"\n✅ Exported {len(high_risk_idle)} high-risk idle opps to: high_risk_idle_opps.csv")

# Save full enriched dataset
df.to_csv('stage_velocity_enriched.csv', index=False)
print(f"✅ Exported full enriched dataset ({len(df)} opps) to: stage_velocity_enriched.csv")


✅ Exported 635 high-risk idle opps to: high_risk_idle_opps.csv
✅ Exported full enriched dataset (2473 opps) to: stage_velocity_enriched.csv


## 8. Same Quarter Close & Fast Close Analysis

Analyzing the proportion of opportunities that close quickly:
- **Same Quarter Close**: Created and closed within the same fiscal quarter
- **Fast Close**: Created and closed within 60 days

In [227]:
# ==================== SAME QUARTER CLOSE & FAST CLOSE ANALYSIS ====================
# Filter to closed opps only
closed_analysis_df = df[df['OUTCOME'].isin(['WON', 'LOST'])].copy()

print("="*80)
print("⚡ SAME QUARTER CLOSE & FAST CLOSE ANALYSIS")
print("="*80)

# Overall metrics
same_qtr_rate = closed_analysis_df['SAME_QTR_CLOSE'].mean() * 100
fast_close_rate = closed_analysis_df['FAST_CLOSE'].mean() * 100

print(f"\n📊 Overall Metrics (n={len(closed_analysis_df):,} closed opps):")
print(f"   • Same Quarter Close Rate: {same_qtr_rate:.1f}%")
print(f"   • Fast Close Rate (≤60 days): {fast_close_rate:.1f}%")

# By Outcome
print(f"\n📊 By Outcome:")
for outcome in ['WON', 'LOST']:
    subset = closed_analysis_df[closed_analysis_df['OUTCOME'] == outcome]
    print(f"   {outcome}:")
    print(f"      Same Qtr Close: {subset['SAME_QTR_CLOSE'].mean()*100:.1f}% ({subset['SAME_QTR_CLOSE'].sum():,}/{len(subset):,})")
    print(f"      Fast Close:     {subset['FAST_CLOSE'].mean()*100:.1f}% ({subset['FAST_CLOSE'].sum():,}/{len(subset):,})")

# ==================== VISUALIZATION 1: Same Quarter Close Rate by Created Quarter ====================
same_qtr_by_quarter = closed_analysis_df.groupby(['CREATED_QUARTER', 'OUTCOME']).agg({
    'SAME_QTR_CLOSE': ['sum', 'count', 'mean']
}).reset_index()
same_qtr_by_quarter.columns = ['CREATED_QUARTER', 'OUTCOME', 'SAME_QTR_COUNT', 'TOTAL_COUNT', 'SAME_QTR_RATE']
same_qtr_by_quarter['SAME_QTR_RATE'] = same_qtr_by_quarter['SAME_QTR_RATE'] * 100

fig = px.bar(
    same_qtr_by_quarter,
    x='CREATED_QUARTER',
    y='SAME_QTR_RATE',
    color='OUTCOME',
    color_discrete_map=COLOR_MAP,
    title='📅 Same Quarter Close Rate by Created Quarter & Outcome',
    labels={'SAME_QTR_RATE': 'Same Quarter Close %', 'CREATED_QUARTER': 'Created Quarter'},
    barmode='group',
    text='SAME_QTR_COUNT'
)
fig.update_layout(template=PLOT_TEMPLATE)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.show()

# ==================== VISUALIZATION 2: Fast Close Rate by Created Quarter ====================
fast_close_by_quarter = closed_analysis_df.groupby(['CREATED_QUARTER', 'OUTCOME']).agg({
    'FAST_CLOSE': ['sum', 'count', 'mean']
}).reset_index()
fast_close_by_quarter.columns = ['CREATED_QUARTER', 'OUTCOME', 'FAST_CLOSE_COUNT', 'TOTAL_COUNT', 'FAST_CLOSE_RATE']
fast_close_by_quarter['FAST_CLOSE_RATE'] = fast_close_by_quarter['FAST_CLOSE_RATE'] * 100

fig2 = px.bar(
    fast_close_by_quarter,
    x='CREATED_QUARTER',
    y='FAST_CLOSE_RATE',
    color='OUTCOME',
    color_discrete_map=COLOR_MAP,
    title='⚡ Fast Close Rate (≤60 days) by Created Quarter & Outcome',
    labels={'FAST_CLOSE_RATE': 'Fast Close %', 'CREATED_QUARTER': 'Created Quarter'},
    barmode='group',
    text='FAST_CLOSE_COUNT'
)
fig2.update_layout(template=PLOT_TEMPLATE)
fig2.update_traces(texttemplate='%{text}', textposition='outside')
fig2.show()

⚡ SAME QUARTER CLOSE & FAST CLOSE ANALYSIS

📊 Overall Metrics (n=1,539 closed opps):
   • Same Quarter Close Rate: 39.4%
   • Fast Close Rate (≤60 days): 46.8%

📊 By Outcome:
   WON:
      Same Qtr Close: 52.5% (309/589)
      Fast Close:     57.0% (336/589)
   LOST:
      Same Qtr Close: 31.3% (297/950)
      Fast Close:     40.5% (385/950)


In [228]:
# ==================== VISUALIZATION 3: Donut Chart - Overall Breakdown ====================
# Create a 2x2 subplot for donut charts
fig3 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=('Same Quarter Close', 'Fast Close (≤60 days)')
)

# Same Quarter Close donut
same_qtr_counts = closed_analysis_df['SAME_QTR_CLOSE'].value_counts()
fig3.add_trace(
    go.Pie(
        labels=['Different Quarter', 'Same Quarter'],
        values=[same_qtr_counts.get(False, 0), same_qtr_counts.get(True, 0)],
        hole=0.5,
        marker_colors=['#EF553B', '#00CC96'],
        textinfo='percent+value'
    ),
    row=1, col=1
)

# Fast Close donut
fast_close_counts = closed_analysis_df['FAST_CLOSE'].value_counts()
fig3.add_trace(
    go.Pie(
        labels=['Slow Close (>60d)', 'Fast Close (≤60d)'],
        values=[fast_close_counts.get(False, 0), fast_close_counts.get(True, 0)],
        hole=0.5,
        marker_colors=['#EF553B', '#00CC96'],
        textinfo='percent+value'
    ),
    row=1, col=2
)

fig3.update_layout(
    title='📊 Close Velocity Distribution (All Closed Opps)',
    template=PLOT_TEMPLATE,
    showlegend=True
)
fig3.show()

# ==================== VISUALIZATION 4: Cross-tabulation Heatmap ====================
# Create a 2x2 matrix: Same Qtr vs Fast Close
cross_tab = pd.crosstab(
    closed_analysis_df['SAME_QTR_CLOSE'].map({True: 'Same Quarter', False: 'Different Quarter'}),
    closed_analysis_df['FAST_CLOSE'].map({True: 'Fast (≤60d)', False: 'Slow (>60d)'})
)

fig4 = px.imshow(
    cross_tab,
    text_auto=True,
    color_continuous_scale='Greens',
    title='🔥 Cross-Tab: Same Quarter Close vs Fast Close',
    labels={'x': 'Close Speed', 'y': 'Quarter Alignment', 'color': 'Count'}
)
fig4.update_layout(template=PLOT_TEMPLATE)
fig4.show()

print("\n📋 Cross-Tabulation:")
print(cross_tab)
print(f"\n💡 Insight: {cross_tab.loc['Same Quarter', 'Fast (≤60d)']:,} opps are BOTH same-quarter AND fast-close")
print(f"   This represents {cross_tab.loc['Same Quarter', 'Fast (≤60d)'] / len(closed_analysis_df) * 100:.1f}% of all closed opps")


📋 Cross-Tabulation:
FAST_CLOSE         Fast (≤60d)  Slow (>60d)
SAME_QTR_CLOSE                             
Different Quarter          179          754
Same Quarter               542           64

💡 Insight: 542 opps are BOTH same-quarter AND fast-close
   This represents 35.2% of all closed opps


In [229]:
# ==================== VISUALIZATION 5: Win Rate by Close Velocity ====================
# Does closing faster correlate with winning?

velocity_segments = []
for same_qtr in [True, False]:
    for fast in [True, False]:
        subset = closed_analysis_df[(closed_analysis_df['SAME_QTR_CLOSE'] == same_qtr) &
                                     (closed_analysis_df['FAST_CLOSE'] == fast)]
        if len(subset) > 0:
            segment_name = f"{'Same Qtr' if same_qtr else 'Diff Qtr'} + {'Fast' if fast else 'Slow'}"
            win_rate = (subset['OUTCOME'] == 'WON').mean() * 100
            velocity_segments.append({
                'Segment': segment_name,
                'Win Rate': win_rate,
                'Count': len(subset),
                'Same Quarter': 'Yes' if same_qtr else 'No',
                'Fast Close': 'Yes' if fast else 'No'
            })

velocity_df = pd.DataFrame(velocity_segments)

fig5 = px.bar(
    velocity_df,
    x='Segment',
    y='Win Rate',
    color='Win Rate',
    color_continuous_scale='RdYlGn',
    title='🏆 Win Rate by Close Velocity Segment',
    text='Count',
    labels={'Win Rate': 'Win Rate %'}
)
fig5.update_layout(template=PLOT_TEMPLATE)
fig5.update_traces(texttemplate='n=%{text}', textposition='outside')
fig5.show()

print("\n📋 Win Rate by Close Velocity Segment:")
print(velocity_df.to_string(index=False))

# Key insight
best_segment = velocity_df.loc[velocity_df['Win Rate'].idxmax()]
worst_segment = velocity_df.loc[velocity_df['Win Rate'].idxmin()]
print(f"\n💡 KEY INSIGHTS:")
print(f"   • Best Win Rate: {best_segment['Segment']} ({best_segment['Win Rate']:.1f}%)")
print(f"   • Worst Win Rate: {worst_segment['Segment']} ({worst_segment['Win Rate']:.1f}%)")
print(f"   • Gap: {best_segment['Win Rate'] - worst_segment['Win Rate']:.1f} percentage points")


📋 Win Rate by Close Velocity Segment:
        Segment  Win Rate  Count Same Quarter Fast Close
Same Qtr + Fast 51.845018    542          Yes        Yes
Same Qtr + Slow 43.750000     64          Yes         No
Diff Qtr + Fast 30.726257    179           No        Yes
Diff Qtr + Slow 29.840849    754           No         No

💡 KEY INSIGHTS:
   • Best Win Rate: Same Qtr + Fast (51.8%)
   • Worst Win Rate: Diff Qtr + Slow (29.8%)
   • Gap: 22.0 percentage points
